# DLC: Build Centered Parquets

Load DLC inference outputs, center coordinates on a reference landmark, and write whole-session parquets.

Outputs go to `stroke_orofacial_pipeline_outputs/dlc_analysis/centered_{mode}/` on MICROSCOPE. All three centering modes are built by default.

**Phase 6.7 update:** orchestration now flows through `SessionManifest`. Cell 3 builds one manifest per (animal, date); cell 4 passes each manifest to `build_centered_parquet()`. Phase 4 is the first DLC-required orchestrator — manifests with no DLC files raise a clear FileNotFoundError naming the session, surfaced in the per-session error log.

**See also:** `stroke_orofacial.dlc_io.centering.load_centered_fresh(h5_path, center_mode, likelihood_threshold)` is an exploration escape hatch that returns the same centered DataFrame *without* writing to disk — use it to sweep `likelihood_threshold` interactively without rebuilding the on-disk parquet cache.

In [ ]:
# ============================================================
# USER INPUTS — edit variables below, then run all cells.
# ============================================================

ANIMALS = ["PS46", "PS47", "PS48", "PS49", "PS50", "PS51", "PS54", "PS55"]

# Dates to process per animal. Set to None to process all dates in the
# animal's session range (from configs/animals.yaml).
DATES = None

# Centering modes to build (all three by default)
CENTER_MODES = ["nose", "spout_midpoint", "eye_midpoint"]

# Overwrite mode: "skip" (default) or "force"
OVERWRITE = "skip"

In [ ]:
import datetime
import logging

from stroke_orofacial.animals import load_animals
from stroke_orofacial.config_loader import load_defaults
from stroke_orofacial.dlc_io.parquet import (
    _save_dlc_session_params,
    build_centered_parquet,
)
from stroke_orofacial.manifest import build_manifest
from stroke_orofacial.paths import PathResolver

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

resolver = PathResolver()
animals_config = load_animals()
defaults = load_defaults()
dlc_params = defaults.get("dlc_centering", {})
output_root = resolver.pipeline_outputs("dlc_analysis")

In [ ]:
# ============================================================
# Build a SessionManifest per (animal, date).
# Surfaces DLC h5 discovery in the notebook state. Phase 4 needs DLC,
# so manifests with empty dlc_h5_files will fail at process time —
# easier to filter them up-front here.
# ============================================================

manifests = []  # list of (animal_id, date, manifest)

for animal_id in ANIMALS:
    info = animals_config[animal_id]

    if DATES is not None:
        dates = DATES
    else:
        n_days = (info.session_end - info.session_start).days + 1
        dates = [info.session_start + datetime.timedelta(days=i) for i in range(n_days)]

    for date in dates:
        manifest = build_manifest(animal_id, date, resolver, info)
        manifests.append((animal_id, date, manifest))

n_with_dlc = sum(1 for _, _, m in manifests if m.dlc_h5_files)
n_without_dlc = sum(1 for _, _, m in manifests if not m.dlc_h5_files)
print(f"Built {len(manifests)} manifests:")
print(f"  with DLC files (will process):    {n_with_dlc}")
print(f"  without DLC files (will skip):    {n_without_dlc}")

In [ ]:
# ============================================================
# For each (manifest, center_mode), call build_centered_parquet().
# Sessions without DLC raise FileNotFoundError; logged and skipped.
# ============================================================

n_written, n_skipped, n_error = 0, 0, 0

for animal_id, date, manifest in manifests:
    date_str = date.strftime("%Y%m%d")
    any_written = False
    for mode in CENTER_MODES:
        try:
            result = build_centered_parquet(
                manifest, mode, resolver,
                params=dlc_params, overwrite=OVERWRITE,
            )
            if result is not None:
                any_written = True
                n_written += 1
            else:
                n_skipped += 1
        except (FileNotFoundError, RuntimeError) as e:
            n_error += 1
            logging.warning("Error %s/%s/%s: %s", animal_id, date_str, mode, e)

    # Per-session params.json (always written, captures skipped status)
    _save_dlc_session_params(
        output_root, animal_id, date_str, CENTER_MODES,
        dlc_params, skipped=(not any_written),
    )

print(f"\nDone: Written {n_written}, Skipped {n_skipped}, Errors {n_error}")